In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from data_generation import generate_data
from tokenizer import tokenizer
from train import train

print(torch.cuda.is_available())


In [ ]:
# Define shared helpers for checkpoints, tokenizers, cached training, and plotting.
def final_checkpoint(run_dir, epochs):
    return Path(run_dir) / f"ckpt_{epochs}_epochs.pt"


def build_tokenizer(*splits):
    tok = tokenizer()
    all_data = []
    for split in splits:
        all_data.extend(split)
    tok.build_vocab(all_data)
    return tok


def normalize_train_return(train_result):
    if isinstance(train_result, tuple):
        return train_result[0]
    return train_result


def train_or_load_model(
    *,
    p,
    seed,
    output_dir,
    epochs=500,
    train_split=0.64,
    val_split=0.16,
    data_dir="add_subtract",
    data_seed=42,
    ops=("+", "-"),
):
    run_dir = Path(output_dir)
    train_set, val_set, test_set = generate_data(
        p=p,
        train_split=train_split,
        val_split=val_split,
        ops=list(ops),
        seed=data_seed,
        output_dir=data_dir,
    )
    tok = build_tokenizer(train_set, val_set, test_set)
    metrics_path = run_dir / "metrics.json"
    ckpt_path = final_checkpoint(run_dir, epochs)

    if metrics_path.exists() and ckpt_path.exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
    else:
        metrics = normalize_train_return(train(
            tok=tok,
            train_data=train_set,
            val_data=val_set,
            batch_size=512,
            block_size=16,
            n_layer=2,
            n_head=4,
            n_embd=128,
            dropout=0,
            bias=True,
            learning_rate=1e-3,
            epochs=epochs,
            seed=seed,
            output_dir=str(run_dir),
            intermediate_ckpt=False,
            verbose=True,
            weight_decay=1.0,
            beta1=0.9,
            beta2=0.98,
            warmup_steps=10,
        ))

    return {
        "label": f"p={p}, seed={seed}",
        "p": p,
        "seed": seed,
        "epochs": epochs,
        "run_dir": str(run_dir),
        "metrics": metrics,
        "train_set": train_set,
        "val_set": val_set,
        "test_set": test_set,
    }


def plot_metric_group(results, metric_group, title):
    fig, axs = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(title)

    for result in results:
        metrics = result["metrics"]["combined"] if metric_group == "combined" else result["metrics"]["by_operator"][metric_group]
        x = np.arange(1, len(metrics["train_losses"]) + 1)
        label = result["label"]

        axs[0].plot(x, metrics["train_losses"], label=f"{label} train")
        axs[0].plot(x, metrics["val_losses"], linestyle="--", label=f"{label} val")
        axs[1].plot(x, metrics["train_accs"], label=f"{label} train")
        axs[1].plot(x, metrics["val_accs"], linestyle="--", label=f"{label} val")

    axs[0].set_title("Loss vs Epoch")
    axs[0].set_xlabel("Epoch")
    axs[0].set_ylabel("Loss")
    axs[0].legend(fontsize=8)

    axs[1].set_title("Accuracy vs Epoch")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("Accuracy")
    axs[1].legend(fontsize=8)

    fig.tight_layout()
    plt.show()


In [ ]:
# Train or load four addition/subtraction models: two seeds for p=97 and two seeds for p=113.
EXPERIMENTS = [
    {"p": 97, "seed": 42, "output_dir": "add_subtract/p97_seed42"},
    {"p": 97, "seed": 43, "output_dir": "add_subtract/p97_seed43"},
    {"p": 113, "seed": 42, "output_dir": "add_subtract/p113_seed42"},
    {"p": 113, "seed": 43, "output_dir": "add_subtract/p113_seed43"},
]

experiment_results = [train_or_load_model(**config) for config in EXPERIMENTS]


In [ ]:
# Plot combined train/validation loss and accuracy for all four models.
plot_metric_group(
    experiment_results,
    metric_group="combined",
    title="Combined Addition/Subtraction Metrics",
)


In [ ]:
# Plot addition-only train/validation loss and accuracy for all four models.
plot_metric_group(
    experiment_results,
    metric_group="+",
    title="Addition-Only Metrics",
)


In [ ]:
# Plot subtraction-only train/validation loss and accuracy for all four models.
plot_metric_group(
    experiment_results,
    metric_group="-",
    title="Subtraction-Only Metrics",
)
